# Image-Based Fire Risk Assessment with Claude

This notebook sets up a multimodal workflow to analyze satellite imagery and produce a structured wildfire risk assessment for a residential property.

In [1]:
# Load env variables and create client
import base64
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

## Environment Setup

Load environment variables and initialize the Anthropic client and model for multimodal image analysis.

In [2]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024,
):
    params = {
        "model": model,
        "max_tokens": 4000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

## Fire Risk Evaluation Prompt

Create a detailed instruction prompt to guide visual analysis of residence location, tree overhang, defensible space, and final risk scoring.

In [3]:
# Fire risk assessment prompt
prompt = """
Analyze the attached satellite image of a property with these specific steps:

1. Residence identification: Locate the primary residence on the property by looking for:
   - The largest roofed structure 
   - Typical residential features (driveway connection, regular geometry)
   - Distinction from other structures (garages, sheds, pools)
   Describe the residence's location relative to property boundaries and other features.

2. Tree overhang analysis: Examine all trees near the primary residence:
   - Identify any trees whose canopy extends directly over any portion of the roof
   - Estimate the percentage of roof covered by overhanging branches (0-25%, 25-50%, 50-75%, 75-100%)
   - Note particularly dense areas of overhang

3. Fire risk assessment: For any overhanging trees, evaluate:
   - Potential wildfire vulnerability (ember catch points, continuous fuel paths to structure)
   - Proximity to chimneys, vents, or other roof openings if visible
   - Areas where branches create a "bridge" between wildland vegetation and the structure
   
4. Defensible space identification: Assess the property's overall vegetative structure:
   - Identify if trees connect to form a continuous canopy over or near the home
   - Note any obvious fuel ladders (vegetation that can carry fire from ground to tree to roof)

5. Fire risk rating: Based on your analysis, assign a Fire Risk Rating from 1-4:
   - Rating 1 (Low Risk): No tree branches overhanging the roof, good defensible space around the structure
   - Rating 2 (Moderate Risk): Minimal overhang (<25% of roof), some separation between tree canopies
   - Rating 3 (High Risk): Significant overhang (25-50% of roof), connected tree canopies, multiple points of vulnerability
   - Rating 4 (Severe Risk): Extensive overhang (>50% of roof), dense vegetation against structure, numerous ember catch points, limited defensible space

For each item above (1-5), write one sentence summarizing your findings, with your final response being the numeric Fire Risk Rating (1-4) with a brief justification.
"""

## Next Step: Attach Image and Run Inference

Load the image file as base64 content, attach it to the user message with the prompt, and send the multimodal request to Claude.

In [9]:
from pathlib import Path
import mimetypes

candidate_dirs = [Path("data/images"), Path("../data/images")]
image_candidates = []

for images_dir in candidate_dirs:
    if images_dir.exists() and images_dir.is_dir():
        image_candidates = sorted(
            [
                p
                for p in images_dir.glob("*")
                if p.is_file()
                and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp", ".gif"}
            ]
        )
        if image_candidates:
            break

if not image_candidates:
    searched = ", ".join(str(p) for p in candidate_dirs)
    raise FileNotFoundError(
        f"No supported image files found. Searched: {searched}"
    )

image_path = image_candidates[0]
media_type, _ = mimetypes.guess_type(image_path.name)
if media_type is None:
    media_type = "image/jpeg"

with open(image_path, "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

messages = []
add_user_message(
    messages,
    [
        {"type": "text", "text": prompt},
        {
            "type": "image",
            "source": {
                "type": "base64",
                "media_type": media_type,
                "data": image_data,
            },
        },
    ],
)

response = chat(messages)

print(f"Analyzed image: {image_path}")
print(text_from_message(response))

Analyzed image: ../data/images/prop1.png
# Satellite Image Fire Risk Analysis

## 1. Residence Identification
The primary residence is a large rectangular structure with a light-colored roof located in the upper-center portion of the property, featuring what appears to be a driveway access and surrounded by dense vegetation on all sides.

## 2. Tree Overhang Analysis
Multiple mature trees with substantial canopies directly overhang the residence roof, with estimated coverage of 50-75% of the total roof area, particularly concentrated on the eastern and southern portions of the structure where dense, dark canopy shadows are clearly visible over the roofline.

## 3. Fire Risk Assessment
The overhanging trees create severe wildfire vulnerability through multiple ember catch points across the majority of the roof surface, with branches forming direct fuel bridges from surrounding vegetation to the structure and minimal visible clearance between tree canopies and the roof.

## 4. Defensible